<div style="background-color:#f0f8ff; padding:12px; border-radius:8px;">
  <b>Demo A — Public API → JSON → NumPy (Bangkok forecast via Open-Meteo)</b>
</div>

**Question:** What will Bangkok's hourly temperature look like over one forecast day?

**Learning path:** construct an API request → check the HTTP response → inspect JSON → validate the observations → convert them to NumPy.


### Step 1 — Request data from a public API

Identify the **endpoint** and **parameters** before running the cell. Open-Meteo does not require an API key for this example.


In [ ]:
import requests

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 13.7563,
    "longitude": 100.5018,
    "hourly": "temperature_2m",
    "forecast_days": 1,
    "timezone": "Asia/Bangkok",
}

response = requests.get(url, params=params, timeout=15)
print("Request URL:", response.url)
print("HTTP status:", response.status_code)
response.raise_for_status()

js = response.json()
print(js.keys())

In [ ]:
js['hourly']

In [ ]:
print("Top-level keys:", sorted(js.keys()))
print("Hourly fields:", list(js["hourly"].keys()))
print("Units:", js["hourly_units"])

for timestamp, temperature in zip(
    js["hourly"]["time"][:3],
    js["hourly"]["temperature_2m"][:3],
):
    print(timestamp, "→", temperature, js["hourly_units"]["temperature_2m"])


### Step 3 — Validate and convert the observations to NumPy

The timestamps and measurements must have matching shapes. Missing temperatures are represented as `NaN` and excluded from the summary statistics.


In [ ]:
import numpy as np

hourly = js["hourly"]
times = np.array(hourly["time"], dtype="datetime64[m]")
valid_temps = np.array(
    [np.nan if value is None else value for value in hourly["temperature_2m"]],
    dtype=float,
)

unit = js["hourly_units"]["temperature_2m"]
print(
    f"samples: {valid_temps.size}, "
    f"Mean temperature: {valid_temps.mean():.2f} {unit}, "
    f"Std: {valid_temps.std():.2f}, "
    f"Min: {valid_temps.min():.2f}, "
    f"Max: {valid_temps.max():.2f}"
)


### Quick exercise

1. Change the coordinates to another city and predict whether the mean will rise or fall.
2. Add `relative_humidity_2m` to the `hourly` parameter.
3. Verify that time, temperature and humidity contain the same number of observations.


<div style="background-color:#f0f8ff; padding:12px; border-radius:8px;">
  <b>Demo B — Authenticated GitHub API → JSON records → NumPy</b>
</div>

**Question:** Among ten popular Python repositories returned by GitHub, how do their stars, forks and open issues compare?

**Learning path:** configure optional authentication → request one page → inspect the JSON records → extract selected fields → build a two-dimensional NumPy array.


### Step 1 — Configure authentication and the request

The GitHub token is read from the `GITHUB_TOKEN` environment variable; it is never printed or stored directly in the notebook. The public request can run without a token, while an authenticated request generally has a larger request allowance. This demonstration requests only ten repositories from one page.


In [ ]:
import os
import requests
import numpy as np

token = os.getenv("GITHUB_TOKEN")

headers = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
    "User-Agent": "DSClass/1.0",
}
if token:
    headers["Authorization"] = f"Bearer {token}"

url = "https://api.github.com/search/repositories"
params = {
    "q": "language:python stars:>10000",
    "sort": "stars",
    "order": "desc",
    "per_page": 10,
}

print("Authentication:", "token configured" if token else "public/unauthenticated")
print("Query:", params["q"])


### Step 2 — Request and inspect one page

Send one request and convert the response from JSON into a Python dictionary. The actual repository records are stored in the `items` list. Compare `total_count` with the number of records returned on this page.


In [ ]:
response = requests.get(url, headers=headers, params=params, timeout=30)
print("Request URL:", response.url)
print("HTTP status:", response.status_code)
response.raise_for_status()

js = response.json()
repos = js.get("items", [])
if not repos:
    raise ValueError("GitHub returned no repositories for this query.")

print("Top-level keys:", sorted(js.keys()))
print("Total matching repositories:", js.get("total_count"))
print("Repositories returned:", len(repos))
print("First repository:", repos[0]["full_name"])
print("Rate-limit requests remaining:", response.headers.get("X-RateLimit-Remaining"))


In [ ]:
js['items']

### Step 3 — Extract selected fields and build a NumPy array

Each repository is a dictionary. Extract its name separately, then create one numeric row containing stars, forks and open issues. `np.array()` combines the rows into a matrix with shape `(repositories, metrics)`.


In [ ]:
names = [repo["full_name"] for repo in repos]

X = np.array(
    [
        [
            repo["stargazers_count"],
            repo["forks_count"],
            repo["open_issues_count"],
        ]
        for repo in repos
    ],
    dtype=np.int64,
)

print("Array shape:", X.shape)
print("Columns: stars, forks, open issues")

for name, (stars, forks, issues) in zip(names, X):
    print(f"{name}: stars={stars:,}, forks={forks:,}, open issues={issues:,}")

print(
    f"\nMean stars: {X[:, 0].mean():.2f}, "
    f"Mean forks: {X[:, 1].mean():.2f}, "
    f"Mean open issues: {X[:, 2].mean():.2f}"
)


<div style="background-color:#f0f8ff; padding:12px; border-radius:8px;">
  <b>Demo C — Read Parquet from S3 → NumPy</b>
</div>

**Question:** How can we acquire selected columns from a partitioned Parquet dataset in cloud object storage and calculate total sales revenue?

**Learning path:** prepare a small Parquet file → identify the S3 bucket and partition → read selected columns → inspect and validate the schema → convert to NumPy → calculate revenue.


In [ ]:
import pandas as pd

sales = pd.DataFrame({
    "product": ["Laptop", "Mouse", "Keyboard", "Monitor"],
    "qty": [2, 10, 5, 3],
    "price": [30000, 500, 1200, 7500],
})

sales.to_parquet("sales_demo.parquet", index=False)
sales

### Step 1 — Configure the S3 source

The default source is `s3://chantri-cpdsai/sales_demo.parquet`. A private bucket uses credentials from an AWS profile, environment, IAM role or another configured credential provider. For a genuinely public bucket, set `S3_ANONYMOUS=1`. `S3_PARQUET_PATH` can still override the default when needed.


In [ ]:
import os

default_path = "s3://chantri-cpdsai/sales_demo.parquet"
path = os.getenv("S3_PARQUET_PATH", default_path)

anonymous = os.getenv("S3_ANONYMOUS") == "1"
storage_options = {"anon": True} if anonymous else None

print("S3 source:", path)
print("Access mode:", "anonymous/public" if anonymous else "configured AWS credentials")


### Step 2 — Read selected Parquet columns

`columns=[...]` is **column projection**: only the requested Parquet columns are acquired. This is different from predicate pushdown, which uses `filters=` to reduce rows or partitions.


In [ ]:
import pandas as pd

df = pd.read_parquet(
    path,
    columns=["product", "qty", "price"],
    storage_options=storage_options,
)

print("Rows and columns:", df.shape)
print("Columns:", df.columns.tolist())
print("Data types:")
print(df.dtypes)
print("Preview:")
print(df.head().to_string(index=False))


### Step 3 — Validate and clean the acquired records

Confirm that the expected schema is available, convert the numeric fields explicitly, and remove records that cannot be used in the calculation.


In [ ]:
required = {"product", "qty", "price"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

sales = df[["product", "qty", "price"]].copy()
sales[["qty", "price"]] = sales[["qty", "price"]].apply(
    pd.to_numeric,
    errors="coerce",
)
sales = sales.dropna(subset=["qty", "price"])

if sales.empty:
    raise ValueError("No valid qty/price rows remained after cleaning.")

print("Valid rows:", len(sales))


### Step 4 — Convert the numeric fields to NumPy and calculate revenue

Rows represent sales records. The two NumPy columns are quantity and unit price. `np.dot(quantity, price)` calculates the sum of `quantity × price` across all rows.


In [ ]:
import numpy as np

X = sales[["qty", "price"]].to_numpy(dtype=float)
row_revenue = X[:, 0] * X[:, 1]
total_revenue = np.dot(X[:, 0], X[:, 1])

result = sales.copy()
result["revenue"] = row_revenue

print("Array shape:", X.shape)
print(result.to_string(index=False))
print(f"Total revenue: {total_revenue:,.2f} THB")


### Quick student exercise

1. Add `filters=[("price", ">=", 1000)]` and compare the acquired rows.
2. Verify that `row_revenue.sum()` and `np.dot(X[:, 0], X[:, 1])` produce the same total.


In [ ]:
X = df[["qty", "price"]].to_numpy(dtype=float)

row_revenue = X[:, 0] * X[:, 1]

total_1 = row_revenue.sum()
total_2 = np.dot(X[:, 0], X[:, 1])

print("Using sum:", total_1)
print("Using dot product:", total_2)
print("Same result:", np.isclose(total_1, total_2))

assert np.isclose(total_1, total_2)

<div style="background-color:#f0f8ff; padding:12px; border-radius:8px;">
  <b>Demo E — Text dataset → NumPy (length stats)</b>
</div>

In [ ]:
from datasets import load_dataset
import numpy as np


# IMDb sentiment (pin revision in real projects)
ds = load_dataset("imdb")

print(ds)       #text: the movie review
print(ds.keys()) #label: 0 for negative and 1 for positive


train = ds["train"].shuffle(seed=42).select(range(2000)) #Selects the first 2,000 records from the training split.
texts = train["text"]
labels = np.array(train["label"], dtype=np.int8)

#print(texts[0][:300]) #displays the first 300 characters of one review.
#print(labels[:10])

lengths = np.array(
    [len(text.split()) for text in texts],
    dtype=np.int32,
)
print(f"n: {lengths.size:,}, "
    f"mean length: {lengths.mean():.2f} words, "
    f"positive rate: {labels.mean():.2%}"
)

In [ ]:
print(type(texts[0]))

In [ ]:
print(texts)

In [ ]:
df = train.to_pandas()
print(df)

<div style="background-color:#f0f8ff; padding:12px; border-radius:8px;">
  <b>Demo F — Image dataset → NumPy (pixel means)</b>
</div>

In [ ]:
from datasets import load_dataset
import numpy as np

# Acquire 2,000 MNIST training examples
mnist = load_dataset("mnist")
sample = mnist["train"].shuffle(seed=42).select(range(2000))

# Convert images and labels to NumPy
imgs = [np.array(img) for img in sample["image"]]
X = np.stack(imgs, axis=0).astype("float32") / 255.0
y = np.array(sample["label"], dtype=np.int8)

# Calculate mean brightness per image
per_image_mean = X.mean(axis=(1, 2))

print("X shape:", X.shape)
print("y shape:", y.shape)
print("dtype:", X.dtype)
print(f"pixel range: {X.min():.2f}–{X.max():.2f}")
print(f"mean brightness: {per_image_mean.mean():.4f}")
print(f"brightness std: {per_image_mean.std():.4f}")


<div style="background-color:#f0f8ff; padding:12px; border-radius:8px;">
  <b>Demo G: Generate a tiny labeled dataset (JSON) → NumPy</b>
</div>

In [ ]:
import os, json, numpy as np
from openai import OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


prompt = (
"Generate 5 short product reviews as a JSON array. "
"Each item is an object with keys 'text' (≤20 words) and 'stars' (1–5 integer). "
"Return ONLY valid JSON, no prose."
"Remove json text in the first line."
)


resp = client.responses.create(
    model="gpt-4o-mini",  # Replace with your desired model
    input=[
        {"role": "user", "content": prompt},
    ]
)

try:
    data = json.loads(resp.output_text)
except json.JSONDecodeError:
    print("Output was not valid JSON:", resp.output_text)
    data = None

# Build NumPy features
stars = np.array([int(x["stars"]) for x in data], dtype=np.int8)
lengths = np.array([len(x["text"].split()) for x in data], dtype=np.int16)
print("n:", stars.size, "mean_stars:", stars.mean(), "mean_len:", lengths.mean())

In [ ]:
print(data)